In [13]:
import yunyi
import pandas as pd
from datetime import date
event = '1106' # 云南省围棋精英加星赛【11月赛】 # '1089' # 云南省围棋精英加星赛【10月赛】
# '1074' # 云南省围棋精英加星赛【9月赛】# '1045' # 云南省围棋精英加星赛【8月赛】# '1017' # 云南省围棋精英加星赛【7月赛】 
# '1005' # 云南省围棋精英加星赛【6月赛】# "992" # 云南省围棋精英加星赛【5月赛】 # "983" #- 云南省围棋精英加星赛【4月赛】
# 973 # 云南省围棋精英加星赛【3月赛】 # '961' # 云南省围棋精英加星赛【2月赛】 # "940" # 云南省围棋精英加星赛【1月赛】
# "924" # 云南省围棋精英加星赛【12月赛】# "913" # 云南省围棋精英加星赛【11月赛】 # "903" # 云南省围棋精英加星赛【10月赛】
# "892" # 云南省围棋精英加星赛【9月赛】' # "852" # 云南省围棋精英加星赛【8月赛】# "841" # 云南省围棋精英加星赛【7月赛】 
# "831" # 云南省围棋精英加星赛【6月赛】 # "819" # 5段加星赛【5月赛】 # 810 # 5段加星赛【4月赛】
# " 800" # 5段加星赛【3月赛】# "789" # 5段加星赛【2月赛】 # 772 # 5段加星赛【1月赛】
# "756" # 5段加星赛【12月赛】# "748" - 5段加星赛【11月赛】 # "739" - 五段加星赛10月
season = "202411" or date.strftime(date.today(),"%Y%m")

#event = 850 # 2023年第二季度赛
#season = "2023Q2"

level = 30 # 5 dan 

In [14]:
# %% Load games and players from Yunyi.com
# 3dan to 4dan games = [721, 651, 613, 536, 419, 391]
from IPython.display import clear_output as clearoutput
def clear_output():
    for i in range(5):
        clearoutput(wait=True)
    return

yunyi.token = yunyi.login()
players = yunyi.get_active_players(event)
games = yunyi.get_games(level=level) 
clear_output()
print(yunyi.token)

49cb2031437445249a474cd7e00474c1


In [15]:
# %% convert all games to Excel format (Run ONCE!)
import json
import pandas as pd
try:
    # Update active games
    with open("all_games.json","rb") as fp:
        djo = json.load(fp)
    ev_list = [x['eventId'] for x in djo['data']]
    game_count_before = len(djo['data'])
    print('Total game items: ', len(djo['data']))
    comp_game_jo = yunyi.call_api("/go/event/batchquery?statusList%5B0%5D=3&pageNo=1&pageSize=600&typeList=1&name=")
    for g in comp_game_jo:
        if g['eventId'] not in ev_list:
            djo['data'].append(g)
    djo['totalItems'] = len(djo['data'])
    del ev_list, comp_game_jo

    active_game_jo = yunyi.call_api("/go/event/batchquery?statusList%5B0%5D=2&pageNo=1&pageSize=600&typeList=1&name=")
    if len(active_game_jo)==0: 
        raise UserWarning('No active game found!')
    print([f"'{x['eventId']}' # {x['name']}" for x in active_game_jo], sep='\n')
    if djo['data'][-1]['eventId'] != active_game_jo[-1]['eventId']:
        for jo in djo['data']:
            jo['status']=3
        djo['data'].extend(active_game_jo)
        djo['totalItems'] = len(djo['data'])
    if game_count_before != len(djo['data']):
        with open("all_games.json","w+", encoding='utf-8') as fp:
            json.dump(djo, fp)
    print('Total game items: ', len(djo['data']))
    del djo
    # Update Excel file
    with open("all_games.json","rb") as fp:
        df = pd.json_normalize(json.load(fp)['data'])
        df.drop(['tenantId','groupId','instId','tag','bizType','imgOssKey','enrollEndTime'], axis=1, inplace=True)
        df['startTime'] = pd.to_datetime(df['matchStartTime']).dt.date
        df.drop(['matchStartTime'], axis=1, inplace=True)
        df['endTime'] = pd.to_datetime(df['endTime']).dt.date
        df.to_excel('all_games.xlsx',index=False)
except Exception as ex:
    print(ex)

Total game items:  691
["'1106' # 云南省围棋精英加星赛【11月赛】"]
Total game items:  691


In [16]:
# %% Run once is enough!!! Get all game results.
gids = [g['id'] for g in games]
# add the active event to the list, when it is completed but status not changed yet.
#gids.insert(0, event)
all_results = []
for  g in gids:
    data = yunyi.get_game_results(g)
    if len(data) > 0:
        for x in data:
            x['game_name'] = [g['name'] for g in games if g['id'] == x['game_id']][0]
        all_results.extend(data)
df = pd.json_normalize(all_results)
df.drop_duplicates(inplace=True, ignore_index=True, keep='last')
df.to_excel(f'all_results.xlsx', sheet_name=yunyi.level_map[level], index=False)

In [17]:
# %%
# Run once is enough!!! Get game results of the level.
"""" NOT IN USE AT NOW!
gids = [g['id'] for g in games if g['level']==level]
# add the active event to the list, when it is completed but status not changed yet.
#gids.insert(0, event)
all_results = []
for  g in gids:
    data = yunyi.get_game_results(g)
    if len(data) > 0:
        all_results.extend(data)
df = pd.json_normalize(all_results)
df.to_excel(f'all_results_{yunyi.level_map[level]}.xlsx', sheet_name=yunyi.level_map[level])
"""
clear_output()

In [18]:
# %% Update results of the active game.
ref_level = 31 # 3dan
gids = [g['id'] for g in games if g['level']==ref_level]
players = yunyi.get_active_players(event)
for g in gids:
    data = yunyi.get_game_results(g, players)
df = pd.json_normalize(players)
df.to_excel(f'player_stats_{yunyi.level_map[ref_level]}.xlsx', sheet_name=yunyi.level_map[ref_level])

In [19]:
# %% update game_stats_{event}.xlsx
df = pd.json_normalize(yunyi.get_game_results(event))
df.to_excel(f'game_stats_{event}.xlsx')

In [20]:
# %%
# Get game list (All available, paid = 1)
# https://weiqi-v2.ynwqxh.com/go/event/batchquery?statusList%5B0%5D=1&pageNo=1&pageSize=600&typeList=1&name=

# Get game list (Completed = 3)
# https://weiqi-v2.ynwqxh.com/go/event/batchquery?statusList%5B0%5D=3&statusList%5B1%5D=5&pageNo=1&pageSize=500&typeList=&name=

# Get the games from the number of player
#  https://weiqi-v2.ynwqxh.com/go/arrange/user/arrangevs/batchquery?eventId=722&no=128

# Get a specific game as sgf format
# https://weiqi-v2.ynwqxh.com/go/playchess/match/sgf/download.resource?arrgVsId=177625&tenantId=183&token=458b1ea9d80742d786b6f0b5c9942ab5

In [21]:
# %% Download sgf from all players.
player_id = [g['id'] for g in players] #[1, 2, 3, 4, 5, 6]
#player_id = '194'
#yunyi.token = yunyi.login()
clear_output()
completed_games = yunyi.download_sgf(event=event, player_no=player_id)
df = pd.json_normalize(completed_games).drop_duplicates()
df.to_excel(f'sgf_list_{event}.xlsx')
print ('\n', df.shape)

Other error occurred: HTTPSConnectionPool(host='weiqi-v2.ynwqxh.com', port=443): Max retries exceeded with url: /go/arrange/user/arrangevs/batchquery?eventId=1106&no=86 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000022AD2484580>, 'Connection to weiqi-v2.ynwqxh.com timed out. (connect timeout=None)'))


Download game: yunyi_13_86_300615.sgf ... 52   of     52  
 (46, 17)


In [22]:
# %% sgf_analysis from GoPredict_new
clear_output()
import sys
sys.path.append('../GoPredict')
import GoPredict_new as gpn
gopred = gpn.GoPredict()
games, results = gopred.sgf_analysis(analysis_path=f"sgf_{event}", board_size=19)

if not results is None:
    display(results.loc[results['name']=='Bob Zhao'])

2024-11-16 16:44:54-0800: OpenCL backend thread 0: FP16Storage true FP16Compute false FP16TensorCores true FP16TensorCoresFor1x1 false
2024-11-16 16:44:54-0800: Loaded config C:\Users\xiaoj\.katrain\analysis_config.cfg and/or command-line and query overrides
2024-11-16 16:44:54-0800: Loaded model C:\Users\xiaoj\.katrain\kata1-b28c512nbt-s7168446720-d4316919285.bin.gz
2024-11-16 16:44:54-0800: Config override: homeDataDir = C:\Users\xiaoj/.katrain
2024-11-16 16:44:54-0800: Analyzing up to 16 positions at a time in parallel
2024-11-16 16:44:54-0800: Started, ready to begin handling requests
>>> waiting for engine to finish...0 queries left <<<<<
=== End of analysis ===
>>> Saving game reports... >>>
=== End of reporting === sgf_1106\prediction.xlsx


,game_id,name,color,Re,RankNormalize


In [23]:
# %% Save the predictions to the Excel for analysis
from openpyxl.reader.excel import load_workbook
from openpyxl.workbook import Workbook
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.worksheet.table import Table as xlTable
from openpyxl.cell import Cell
from openpyxl.utils import get_column_letter
import os, shutil
fname = f"sgf_{event}/prediction.xlsx"
root_path = os.path.join(os.environ['USERPROFILE'], "Documents\\Python")

template_path = os.path.join(root_path, f"GOPredict/data/Yunyi/predictionYunyi_template.xlsx")
to_name = os.path.join(root_path, f"GOPredict/data/Yunyi/predictionYunyi_{event}_{season}.xlsx")
wb1:Workbook = load_workbook(fname)
ws1:Worksheet = wb1["data"]
if "Data" not in ws1.tables:
    ws1.add_table(table=xlTable(displayName="Data", ref="A1:" + get_column_letter(ws1.max_column) + str(ws1.max_row)))
    wb1.save(fname)

if not os.path.exists(to_name):
    shutil.copy(template_path, to_name)
wb2:Workbook = load_workbook(to_name)
ws2:Worksheet = wb2["data"]
n = 0
for x in ws1.iter_rows(min_row=2, values_only=True):
    #print(n, ws2.cell(n+2, 1).value)
    if ws2.cell(n+2, 1).value is None:
        #print(ws2.cell(n+2, 1))
        for i in range(len(x)):
            ws2.cell(n+2, i+1).value = x[i]
    else:
        #print(n)
        pass
    n+=1
wb2.save(to_name)

In [24]:
clear_output()
raise SystemExit("Stop here")

SystemExit: Stop here

# _STOP AT HERE!_
### The following blocks are not useful now.

In [ ]:
# %%
""" import shutil
import os
sgf_path = os.path.join(os.path.curdir,f"sgf_{event}")
dst_path = os.path.join("..\GOPredict\data\Yunyi")
def ig_f(dir, files):
    return [f for f in files if os.path.isfile(os.path.join(dst_path, f))]
dst = shutil.copytree(sgf_path, dst_path, symlinks=True, ignore=ig_f, dirs_exist_ok=True)
print(dst) """

In [ ]:
# %%
# Query single player data
import requests
heads = {
    "token": yunyi.token,
    "tenantId" : yunyi.tenant_id
}
query_player=157 
resp = requests.get(f"https://weiqi-v2.ynwqxh.com/go/arrange/user/arrangevs/batchquery?eventId={event}&no={query_player}", headers=heads, verify=yunyi.ca_file)
userData = resp.json()
print(userData['data'][0])

{'arrgVsId': 213141, 'tenantId': 183, 'arrgId': 2070, 'eventId': 841, 'instId': 108, 'exnInf': {}, 'status': 0, 'senteNo': '157', 'senteChessPlayerVO': {'chessPlayerId': 49387, 'userId': 1084, 'groupId': None, 'no': '157', 'initialScore': 0, 'levelScore': 0, 'auxScore': 0, 'exnInf': {'dan_grading': '29', 'cert_id': '39578', 'cert_avatar': 'user/cert/avatar/3184bde483bd4b6680ab7c7033bfecc2.jpg'}, 'eventId': 841, 'tenantId': 183, 'instId': 108, 'name': '魏靖霖', 'phone': '13608862617', 'idcard': '530103200911260319', 'avatar': 'user/avatar/2befcc12495d4ea18b44b7af784fc044.jpg', 'seed': False, 'danGrading': 29, 'danGradingName': '业余4段', 'unit': '个人', 'status': 0, 'type': None, 'teamId': None, 'rank': False, 'score': 0, 'foulNum': 0, 'upNum': 0, 'downNum': 0, 'hold': '0', 'remark': None, 'idType': 1, 'birthDate': '2009-11-26T00:00:00.000+0800', 'gender': 2, 'age': 13, 'gradeVO': None, 'statNo': None, 'teamRelId': None, 'groupName': None, 'teamName': None, 'gmtCreator': 'system', 'gmtCreate': 

In [ ]:
# %%
# %%
# %%
import csv
from openpyxl import load_workbook
from openpyxl.worksheet.table import Table, TableStyleInfo
tableName = 'Data'

style = TableStyleInfo(name="TableStyleMedium9", showFirstColumn=False,
                       showLastColumn=False, showRowStripes=True, showColumnStripes=False)

def colnum_string(n):
    string = ""
    while n > 0:
        n, remainder = divmod(n - 1, 26)
        string = chr(65 + remainder) + string
    return string

wb = load_workbook(filename = 'workbook.xlsx')
ws = wb["inputData"]

with open('input.csv', newline='', encoding='utf-8-sig') as f:
    reader = csv.reader(f, delimiter=';')
    for i, row in enumerate(reader):
        for j, cell in enumerate(row): 
            if not i == 0:
                ws.cell(row=i+1, column=j+1).value = float(cell)
            else:
                ws.cell(row=i+1, column=j+1).value = cell

            maxRef = [i,j]

for i, table in enumerate(ws._tables):
    if table.name == tableName:
        tableRef = i

resTable = Table(displayName="Data", ref="A1:{}{}".format(colnum_string(maxRef[0]), maxRef[1]))
resTable.tableStyleInfo = style

ws._tables[tableRef] = resTable

wb.save('output.xlsx')

In [ ]:
# %%
import requests, pandas as pd, json
with open("c:/Users/xiaoj/Downloads/quanguo_quma.json", "r+", encoding="utf-8") as fp:
    qu = json.load(fp)
df = pd.read_excel("c:/users/xiaoj/Documents/Python/Weiqi/game_stats_819.xlsx")
def code2place(x):
    sheng = x[:2]+'0000'
    shi = x[:4]+'00'
    xian = x[:6]
    result = ''
    if not sheng in qu:
        return 'Unkown'
    result = qu[sheng]
    if shi in qu:
        result = result + qu[shi]
    if xian in qu:
        result = result + qu[xian]
    return result

df['location'] = df['player_idcard'].apply(lambda x: code2place(x))

df.to_excel("c:/users/xiaoj/Documents/Python/Weiqi/game_stats_819_new.xlsx", index=False)

### 给刘老师准备的棋谱 (改名字)

In [ ]:
import glob, os
from katrain.core.game import Game, KaTrainSGF

for filename in glob.glob("../MrLiu/*.sgf"):
    print(filename)
    game = KaTrainSGF.parse_file(filename)
    pb = game.get_property('PB')
    pw = game.get_property('PW')
    pb = pb if pb != 'Bob Zhao' else '赵效京'
    pw = pw if pw != 'Bob Zhao' else '赵效京'
    print(f'{pb}_vs_{pw}.sgf')
    os.rename(filename, f'../MrLiu/{pb}_vs_{pw}.sgf')

../MrLiu\yunyi_101_107_235382.sgf
王浠丞_vs_赵效京.sgf
../MrLiu\yunyi_104_107_235073.sgf
念泽嘉_vs_赵效京.sgf
../MrLiu\yunyi_107_108_234305.sgf
赵效京_vs_刘铭皓.sgf
../MrLiu\yunyi_107_112_235120.sgf
赵效京_vs_冯炳涵.sgf
../MrLiu\yunyi_107_113_234718.sgf
赵效京_vs_董博轩.sgf
../MrLiu\yunyi_107_98_235323.sgf
赵效京_vs_罗逸尘.sgf
../MrLiu\yunyi_110_107_234520.sgf
萧和笙_vs_赵效京.sgf
../MrLiu\yunyi_114_107_234901.sgf
彭劲羽_vs_赵效京.sgf
../MrLiu\yunyi_39_107_235260.sgf
杨钧翔_vs_赵效京.sgf
